In [21]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
JupyterDash.infer_jupyter_proxy_config()

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


from CRUD_Python_Module import AnimalShelter



###########################
# Data Manipulation / Model
###########################
username = "aacuser"
password = "CS340ISFUN"
shelter = AnimalShelter(username, password)


#load data into the dataframe
df = pd.DataFrame.from_records(shelter.read({}))

#remove the id column
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash('SimpleExample')

app.layout = html.Div([
    html.Div(id='hidden-div', style={'display':'none'}),
    #added my name to the title. 
    html.Center(html.B(html.H1("Travis Erwin's Super Cool Dashboard"))),
    html.Center(html.B(html.H2("Selectable, Sortable, Filterable, and Readable!"))),
    html.Hr(),
    
    #create the DataTable
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True, "filter_options":{'case':'insensitive'}} for i in df.columns #added filter_options to eliminate case sensitivity when filtering. 
        ],
        data=df.to_dict('records'),
        
        row_selectable="single",
        selected_rows=[0],
        #pagination
        page_action="native",
        page_current=0,
        page_size=10,  #limits results to 10 per page
        #row sorting
        sort_action="native",
        sort_mode="multi", #added for sorting by multiple categories at once.
        filter_action="native" #filters results by keyword in a column. Filtering caused index errors, had to solve for that in the callback.
        
    ),
    html.Br(),
    html.Hr(),
    html.Div(
            id='map-id',
            className='col s12 m6',
            )
])

#############################################
# Interaction Between Components / Controller
#############################################
#This callback will highlight a row on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    #added if statement to handle the nonetype error I was recieving when starting up my dashboard.
    if selected_columns is None:
        return[]
    
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


#update the map based on the selected row
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
    dff = pd.DataFrame.from_dict(viewData)
    
    #if datatable is empty after filtering, "no data" is printed. 
    if dff.empty:
        return[html.P("No Data")]
    
    #uses first row if nothing is selected, otherwise ensure the selected index is within the index range, this solves the index error I was recieving when filtering. 
    row = 0 if not index else min(index[0], len(dff)-1)
    
    return [
            dl.Map(style={'width':'1000px', 'height': '500px'},
                   center=[30.75,-97.48], zoom=10, children=[dl.TileLayer(id="base-layer-id"),
                                                             dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]],
                                                                       children=[
                                                                           dl.Tooltip(dff.iloc[row,4]),
                                                                           dl.Popup([
                                                                               html.H1("Animal Name"),
                                                                               html.P(dff.iloc[row,9])
                                                                           ])
                                                                       ])
                                                            ])
        ]
     
# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server(port=8051)

Dash app running on https://othellogalileo-idiomaxiom-3000.codio.io/proxy/8051/
